In [1]:
import os, json
import numpy as np, pandas as pd
from sklearn.model_selection import train_test_split, StratifiedKFold
from sklearn.calibration import CalibratedClassifierCV
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import FunctionTransformer
from xgboost import XGBClassifier
import joblib
import sys
sys.path.append("..")
from src.transforms import select_cols_top7, get_preprocessor
from utils import SEED, save_joblib, save_manifest
os.makedirs("../artifacts", exist_ok=True)


In [2]:
df = pd.read_csv("../data/heart_uci_clean.csv")
X = df.drop(columns=['num'])
y = df['num'].values
train_idx = np.load("../artifacts/train_idx.npy")
hold_idx  = np.load("../artifacts/hold_idx.npy")
X_train = X.loc[train_idx]
y_train = y[train_idx]
X_hold  = X.loc[hold_idx]
y_hold  = y[hold_idx]


FileNotFoundError: [Errno 2] No such file or directory: '../artifacts/train_idx.npy'

In [3]:
preproc = joblib.load("../artifacts/preprocessor_initial.joblib")
meta = json.load(open("../artifacts/feature_index_map.json"))
top7_indices = meta["selected_indices"]
def select_top7_portable(X, indices=top7_indices):
    return select_cols_top7(X, indices)
selector_ft = FunctionTransformer(select_top7_portable, validate=False)

xgb = XGBClassifier(use_label_encoder=False, eval_metric='logloss', random_state=SEED, n_estimators=100, learning_rate=0.05, max_depth=3)
pipeline_xgb = Pipeline([("preprocessor", preproc), ("selector", selector_ft), ("model", xgb)])
pipeline_xgb.fit(X_train, y_train)

save_joblib(pipeline_xgb, "../artifacts/xgb_top7_pipeline_baseline.joblib")
save_manifest("../artifacts/manifest.json", {"name":"xgb_top7_baseline","path":"../artifacts/xgb_top7_pipeline_baseline.joblib"})


C:\Users\Nikhil\anaconda3\envs\heartenv\Lib\site-packages\sklearn\base.py:442: InconsistentVersionWarning: Trying to unpickle estimator SimpleImputer from version 1.3.0 when using version 1.7.2. This might lead to breaking code or invalid results. Use at your own risk. For more info please refer to:
https://scikit-learn.org/stable/model_persistence.html#security-maintainability-limitations
  warnings.warn(
C:\Users\Nikhil\anaconda3\envs\heartenv\Lib\site-packages\sklearn\base.py:442: InconsistentVersionWarning: Trying to unpickle estimator StandardScaler from version 1.3.0 when using version 1.7.2. This might lead to breaking code or invalid results. Use at your own risk. For more info please refer to:
https://scikit-learn.org/stable/model_persistence.html#security-maintainability-limitations
  warnings.warn(
C:\Users\Nikhil\anaconda3\envs\heartenv\Lib\site-packages\sklearn\base.py:442: InconsistentVersionWarning: Trying to unpickle estimator Pipeline from version 1.3.0 when using vers

NameError: name 'SEED' is not defined

In [4]:
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=SEED)
# CalibratedClassifierCV can wrap the pipeline directly
cal = CalibratedClassifierCV(base_estimator=pipeline_xgb, cv=cv, method='isotonic')
cal.fit(X_train, y_train)
save_joblib(cal, "../artifacts/xgb_top7_pipeline_calibrated.joblib")
save_manifest("../artifacts/manifest.json", {"name":"xgb_top7_calibrated","path":"../artifacts/xgb_top7_pipeline_calibrated.joblib"})


NameError: name 'SEED' is not defined

In [2]:
from sklearn.metrics import roc_auc_score, average_precision_score, brier_score_loss
def eval_model(m, Xh, yh):
    p = m.predict_proba(Xh)[:,1]
    return {"roc_auc": float(roc_auc_score(yh,p)),
            "avg_precision": float(average_precision_score(yh,p)),
            "brier": float(brier_score_loss(yh,p))}
base_metrics = eval_model(pipeline_xgb, X_hold, y_hold)
cal_metrics  = eval_model(cal, X_hold, y_hold)
import json
json.dump({"baseline":base_metrics,"calibrated":cal_metrics}, open("../artifacts/xgb_top7_metrics.json","w"), indent=2)
print(base_metrics, cal_metrics)


NameError: name 'pipeline_xgb' is not defined